# 01 · Data Cleaning Pipeline
**Payment Failure Intelligence & Revenue Optimization System**

**Goal:** Load raw transaction data, assess quality, fix issues, and export a clean dataset.

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs('../outputs', exist_ok=True)

print("Loading raw data...")
df = pd.read_csv('../data/raw_transactions.csv')
print(f"Shape: {df.shape}")
print(df.dtypes)

## Step 1 — Initial Inspection

In [ ]:
print("\n--- HEAD ---")
print(df.head())

print("\n--- INFO ---")
df.info()

print("\n--- DESCRIBE ---")
print(df.describe())

## Step 2 — Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print("\nMissing Values Report:")
print(missing_report[missing_report['missing_count'] > 0])

# Introduce a few artificial nulls to demonstrate cleaning (realistic scenario)
np.random.seed(0)
null_idx_amount = np.random.choice(df.index, size=200, replace=False)
null_idx_region = np.random.choice(df.index, size=150, replace=False)
null_idx_device = np.random.choice(df.index, size=100, replace=False)

df.loc[null_idx_amount, 'amount']      = np.nan
df.loc[null_idx_region, 'region']      = np.nan
df.loc[null_idx_device, 'device_type'] = np.nan

print(f"\nArtificially introduced nulls:")
print(f"  amount:      {df['amount'].isna().sum()}")
print(f"  region:      {df['region'].isna().sum()}")
print(f"  device_type: {df['device_type'].isna().sum()}")

## Step 3 — Handle Missing Values

In [ ]:
# amount: fill with median (robust to outliers)
median_amount = df['amount'].median()
df['amount'] = df['amount'].fillna(median_amount)
print(f"Filled amount nulls with median: ₹{median_amount:,.2f}")

# region: fill with mode (most common region)
mode_region = df['region'].mode()[0]
df['region'] = df['region'].fillna(mode_region)
print(f"Filled region nulls with mode: {mode_region}")

# device_type: fill with mode
mode_device = df['device_type'].mode()[0]
df['device_type'] = df['device_type'].fillna(mode_device)
print(f"Filled device_type nulls with mode: {mode_device}")

print("\nRemaining nulls:", df.isnull().sum().sum())

## Step 4 — Duplicate Detection & Removal

In [ ]:
dupes_before = len(df)

# Introduce 50 artificial duplicates for demonstration
dup_sample = df.sample(50, random_state=1)
df = pd.concat([df, dup_sample], ignore_index=True)
print(f"Rows before dup removal: {len(df):,}")

df = df.drop_duplicates(subset='transaction_id', keep='first')
print(f"Rows after dup removal : {len(df):,}")
print(f"Duplicates removed      : {len(df) - dupes_before + 50}")

## Step 5 — Timestamp Parsing & Validation

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

invalid_ts = df['timestamp'].isna().sum()
print(f"Invalid timestamps: {invalid_ts}")

# Drop rows with unparseable timestamps
df = df.dropna(subset=['timestamp'])
print(f"Rows after timestamp validation: {len(df):,}")

## Step 6 — Category & Status Validation

In [ ]:
valid_statuses    = ['Success', 'Failed']
valid_categories  = ['Food & Dining', 'Electronics', 'Travel',
                     'Utilities', 'Retail', 'Healthcare', 'Entertainment']
valid_devices     = ['Mobile', 'Desktop', 'Tablet']
valid_regions     = ['North', 'South', 'East', 'West', 'Central']
valid_pm          = ['UPI', 'Credit Card', 'Debit Card', 'Net Banking', 'Wallet']

print("Status values:     ", df['status'].unique())
print("Category values:   ", df['category'].unique())
print("Device values:     ", df['device_type'].unique())
print("Region values:     ", df['region'].unique())
print("Payment methods:   ", df['payment_method'].unique())

# Amount bounds check
print(f"\nAmount range: ₹{df['amount'].min():,.2f} → ₹{df['amount'].max():,.2f}")
df = df[(df['amount'] >= 1) & (df['amount'] <= 500_000)]
print(f"Rows after amount bounds filter: {len(df):,}")

## Step 7 — Export Cleaned Data & Quality Report

In [ ]:
# Reset index
df = df.reset_index(drop=True)

# Save cleaned data
df.to_csv('../outputs/cleaned_data.csv', index=False)
print(f"✓ Saved cleaned_data.csv  ({len(df):,} rows)")

# Data quality report
quality_report = pd.DataFrame({
    'metric': [
        'original_rows',
        'cleaned_rows',
        'duplicates_removed',
        'nulls_imputed_amount',
        'nulls_imputed_region',
        'nulls_imputed_device',
        'invalid_timestamps_dropped',
        'out_of_range_amounts_dropped',
        'overall_failure_rate',
        'date_range_start',
        'date_range_end',
    ],
    'value': [
        100_000,
        len(df),
        50,
        200,
        150,
        100,
        invalid_ts,
        0,
        f"{(df['status'] == 'Failed').mean():.4f}",
        str(df['timestamp'].min().date()),
        str(df['timestamp'].max().date()),
    ]
})
quality_report.to_csv('../outputs/data_quality_report.csv', index=False)
print("✓ Saved data_quality_report.csv")
print("\n=== CLEANING COMPLETE ===")
print(quality_report.to_string(index=False))